# Problem Statement & Objective
Build a conversational chatbot that:

- Remembers conversational context across multiple turns
- Retrieves relevant information from an external knowledge base
- Provides grounded, accurate answers using Retrieval-Augmented Generation (RAG)
- Is deployed as an interactive web application with Streamlit


Key Technologies: LangChain · OpenAI · FAISS · Streamlit · Wikipedia API

**Environment Setup & Imports**

In [3]:
!pip install -q pypdf sentence-transformers faiss-cpu numpy streamlit

In [4]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

**SPLIT TEXT INTO CHUNKS**

In [7]:
reader = PdfReader("/content/sample.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

chunks = [text[i:i+500] for i in range(0, len(text), 500)]

print("Chunks:", len(chunks))

Chunks: 70


**Create Embeddings**

In [8]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**CREATE VECTOR DATABASE (FAISS)**

In [10]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

**RETRIEVAL FUNCTION**

In [11]:
def retrieve(query, k=3):
    q_vec = embedder.encode([query])
    _, indices = index.search(np.array(q_vec), k)

    return [chunks[i] for i in indices[0]]

**SIMPLE MEMORY SYSTEM**

In [12]:
chat_history = []

**CHATBOT ENGINE (RAG + MEMORY)**

In [13]:
def chat(query):
    global chat_history

    docs = retrieve(query)
    context = "\n".join(docs)

    history = "\n".join(chat_history[-5:])

    answer = f"""
Context:
{context}

Conversation History:
{history}

User Question:
{query}

Answer:
Based on the document, the relevant information is above.
"""

    chat_history.append(f"User: {query}")
    chat_history.append(f"Bot: {answer}")

    return answer

**TEST CHATBOT**

In [14]:
print(chat("What is this document about?"))


Context:
reduce conflict in an organization.
3) Discuss the relationship between stress and performance.
4) What do you understand by stress? Discuss various strategies to reduce
stress.
5) ‘Organizations now have multiethnic workforce and hence are vulnerable
to conflicts’. Comment.
15.13 FURTHER READINGS
Bramson, R.(1981). Coping with difficult people. NewYork:Anchor
Cosier,R.A. and Dalton, D.R. Positive Effects of Conflict: A Field Experiment,
International Journal of Conflict Management1(1990): 81-92
52
Interpersonal and Group
Processes UNIT 15 CONFLICT AND STRESS
MANAGEMENT
Objectives
After reading this unit, you should be able to:
• understand the concept of conflict and stress;
• able to identify the causes of conflict in an Organization;
• understand the various styles and of conflict;
• know the strategies to  manage conflict; and
• know how to reduce and manage stress.
Structure
15.1 Introduction
15.2 Understanding the Concept of Conflict
15.3 Types of Conflict
15.4 Causes o

STREAMLIT APP

In [15]:
%%writefile app.py

Writing app.py


In [16]:
import streamlit as st
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

st.title("RAG Chatbot - Task 4")

reader = PdfReader("sample.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

chunks = [text[i:i+500] for i in range(0, len(text), 500)]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

chat_history = []

def retrieve(query, k=3):
    q_vec = embedder.encode([query])
    _, indices = index.search(np.array(q_vec), k)
    return [chunks[i] for i in indices[0]]

def chat(query):
    docs = retrieve(query)
    context = "\n".join(docs)

    return f"Answer based on context:\n\n{context}"

user_input = st.text_input("Ask a question")

if user_input:
    st.write(chat(user_input))

2026-05-24 12:11:56.820 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:11:57.338 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-24 12:11:57.339 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:11:57.340 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-24 12:12:08.597 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:12:08.598 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:12:08.600 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:12:08.601 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-24 12:12:08.601 Session state does not function when running a script without `streamlit run`
2026-05-24 12:12:08.602 Thread 'MainThread': missing ScriptRunContext! This w

In [22]:
%%writefile requirements.txt
streamlit
pypdf
sentence-transformers
faiss-cpu
numpy

Overwriting requirements.txt


**RUN STREAMLIT**

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦your url is: https://dry-carrots-cough.loca.lt
2026-05-24 12:38:32.544 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.182.135.125:8501

